In [16]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openrouter import ChatOpenRouter
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [17]:
load_dotenv()

llm = ChatOpenRouter(
    model='google/gemma-4-26b-a4b-it:free'
)

In [18]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [19]:
def generate_joke(state: JokeState):

    prompt = f'generate joke on the topic {state['topic']}'
    response = llm.ivoke(prompt).content

    return {'joke':response}

In [20]:
def explanation_joke(state: JokeState):

    prompt = f'Write the explanation for the joke - {state['joke']}'

    response = llm.invoke(prompt).content

    return {'explanation':response}

In [21]:
graph = StateGraph(JokeState)

graph.add_node('joke',generate_joke)
graph.add_node('explanation',explanation_joke)

graph.add_edge(START, 'joke')
graph.add_edge('joke', 'explanation')
graph.add_edge('explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [ ]:
config1 = {'configurable': {'thread_id': "1"}}

initial_state = {
    'topic':'indian railway'
}

final_state = workflow.invoke(initial_state, config=config1)

In [ ]:
workflow.get_state(config1)

In [ ]:
list(workflow.get_state_history(config1))